# Notebook 17 — Distributed Inference Concepts

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/17_distributed_inference_concepts.ipynb)

Distributed inference is best learned as a set of practical runtime trade-offs, not as abstract taxonomy. This notebook studies when to shard a model, when to replicate it, when to pipeline it, and when expert routing or prefill-decode separation creates more operational value than raw model scale alone.

## What you will build

- model and hardware fit tables for single-node and distributed deployments
- a tensor-parallel trade-off calculator with communication penalties
- a simple pipeline-parallel bubble simulator
- a data-parallel throughput model for online and offline traffic
- an expert-parallel hotspot simulation and a prefill-decode disaggregation comparison

## Keep the focus practical

This notebook is not trying to derive every parallel algorithm from first principles. The goal is narrower and more useful: understand the runtime consequences of each strategy so you can choose a topology that matches the workload, hardware budget, and latency target.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

## Step 1: Add notebook helpers and an artifact directory

We will use lightweight arithmetic and small simulations so the notebook remains runnable without a cluster.

In [ ]:
random.seed(17)

ARTIFACT_DIR = ARTIFACT_ROOT / "17_distributed_inference_concepts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def percentile(values, pct):
    values = [float(v) for v in values]
    if not values:
        return 0.0
    return float(np.percentile(values, pct))

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Define model and hardware profiles

The model profiles are intentionally coarse. They are detailed enough to reason about memory fit, communication cost, and scaling shape without pretending to be a vendor benchmark.

In [ ]:
model_profiles = [
    {"model": "dense_8b", "params_b": 8, "layers": 32, "hidden_size": 4096, "kv_bytes_per_token": 0.50, "base_decode_tps": 145},
    {"model": "dense_70b", "params_b": 70, "layers": 80, "hidden_size": 8192, "kv_bytes_per_token": 2.90, "base_decode_tps": 46},
    {"model": "moe_180b_16e", "params_b": 180, "layers": 96, "hidden_size": 12288, "kv_bytes_per_token": 3.60, "base_decode_tps": 58},
]

hardware_profiles = [
    {"cluster": "single_h100_80g", "gpus": 1, "gpu_mem_gib": 80, "interconnect": "local", "comm_penalty_ms": 0.0},
    {"cluster": "dual_h100_nvlink", "gpus": 2, "gpu_mem_gib": 80, "interconnect": "nvlink", "comm_penalty_ms": 0.7},
    {"cluster": "eight_h100_nvlink", "gpus": 8, "gpu_mem_gib": 80, "interconnect": "nvlink", "comm_penalty_ms": 0.8},
    {"cluster": "eight_l40s_pcie", "gpus": 8, "gpu_mem_gib": 48, "interconnect": "pcie", "comm_penalty_ms": 2.3},
]

models_df = pd.DataFrame(model_profiles)
hardware_df = pd.DataFrame(hardware_profiles)
models_df

## Step 3: Estimate single-node memory fit

Distributed inference usually starts as a memory problem. If the weights plus a realistic KV cache do not fit with headroom, some form of sharding or offloading becomes mandatory.

In [ ]:
def estimate_weight_gib(params_b, bits=16):
    return params_b * 1_000_000_000 * (bits / 8) / (1024 ** 3)

def estimate_kv_gib(model_row, context_tokens, batch_size, bytes_per_value=2):
    kv_gib = model_row["kv_bytes_per_token"] * context_tokens * batch_size / 1024
    return kv_gib

fit_rows = []
for model in model_profiles:
    for cluster in hardware_profiles:
        weights_gib = estimate_weight_gib(model["params_b"], bits=16)
        kv_gib = estimate_kv_gib(model, context_tokens=8192, batch_size=8)
        available_gib = cluster["gpus"] * cluster["gpu_mem_gib"] * 0.88
        fit_rows.append({
            "model": model["model"],
            "cluster": cluster["cluster"],
            "weights_gib": round(weights_gib, 1),
            "kv_gib_at_8k_x8": round(kv_gib, 1),
            "available_gib": round(available_gib, 1),
            "fits_dense_runtime": weights_gib + kv_gib <= available_gib,
        })

fit_df = pd.DataFrame(fit_rows)
fit_df

## Step 4: Model tensor-parallel trade-offs

Tensor parallelism reduces per-GPU memory pressure but adds communication to every decode step. On strong interconnects this is often worth it. On weak interconnects it can erase the gain.

In [ ]:
def tensor_parallel_plan(model_row, cluster_row, degree):
    weights_per_gpu = estimate_weight_gib(model_row["params_b"], bits=16) / degree
    kv_per_gpu = estimate_kv_gib(model_row, context_tokens=4096, batch_size=6) / degree
    comm_ms = cluster_row["comm_penalty_ms"] * max(degree - 1, 0)
    efficiency = max(0.36, 1.0 - 0.06 * (degree - 1) - 0.04 * (cluster_row["interconnect"] == "pcie") * (degree - 1))
    decode_tps = model_row["base_decode_tps"] * degree * efficiency / (1 + comm_ms / 10)
    return {
        "degree": degree,
        "weights_per_gpu_gib": round(weights_per_gpu, 1),
        "kv_per_gpu_gib": round(kv_per_gpu, 1),
        "comm_ms_per_step": round(comm_ms, 2),
        "decode_tps_est": round(decode_tps, 2),
    }

tp_rows = []
model_row = next(row for row in model_profiles if row["model"] == "dense_70b")
for cluster_name in ["dual_h100_nvlink", "eight_h100_nvlink", "eight_l40s_pcie"]:
    cluster_row = next(row for row in hardware_profiles if row["cluster"] == cluster_name)
    for degree in [1, 2, 4, 8]:
        if degree > cluster_row["gpus"]:
            continue
        tp_rows.append({"cluster": cluster_name, **tensor_parallel_plan(model_row, cluster_row, degree)})

tp_df = pd.DataFrame(tp_rows)
tp_df

## Step 5: Visualize tensor-parallel scaling on different interconnects

The same parallel degree can feel very different on NVLink and PCIe. That is why topology matters as much as raw FLOPs.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for cluster_name, frame in tp_df.groupby("cluster"):
    frame = frame.sort_values("degree")
    ax.plot(frame["degree"], frame["decode_tps_est"], marker="o", label=cluster_name)
ax.set_xlabel("tensor parallel degree")
ax.set_ylabel("estimated decode tokens/s")
ax.set_title("Tensor parallel scaling depends on interconnect quality")
ax.legend()
plt.tight_layout()

## Step 6: Simulate pipeline parallel bubbles

Pipeline parallelism helps when a model is too large to place on one GPU shard without stage partitioning. The cost is bubbles: some stages sit idle while waiting for microbatches to fill the pipe.

In [ ]:
def simulate_pipeline(stage_times_ms, microbatches):
    stage_count = len(stage_times_ms)
    total_ms = sum(stage_times_ms) + max(microbatches - 1, 0) * max(stage_times_ms)
    ideal_ms = microbatches * sum(stage_times_ms)
    bubble_fraction = 1.0 if microbatches == 0 else (stage_count - 1) / (microbatches + stage_count - 1)
    utilization = 1.0 - bubble_fraction
    return {
        "stages": stage_count,
        "microbatches": microbatches,
        "total_ms": round(total_ms, 2),
        "bubble_fraction": round(bubble_fraction, 3),
        "stage_utilization": round(utilization, 3),
        "tokens_per_second_est": round((microbatches * 2048) / max(total_ms / 1000, 0.001), 2),
    }

pipeline_shapes = {
    2: [42, 46],
    4: [21, 24, 25, 19],
    8: [11, 12, 12, 14, 10, 11, 13, 9],
}

pipeline_rows = []
for stages, stage_times in pipeline_shapes.items():
    for microbatches in [1, 2, 4, 8, 16]:
        pipeline_rows.append(simulate_pipeline(stage_times, microbatches))

pipeline_df = pd.DataFrame(pipeline_rows)
pipeline_df

## Step 7: Compare pipeline choices directly

Pipeline parallelism usually needs enough microbatches to hide bubbles. That is one reason it shows up more often in throughput-oriented or offline settings than in low-concurrency interactive traffic.

In [ ]:
pipeline_summary = pipeline_df.pivot(index="microbatches", columns="stages", values="stage_utilization").round(3)
pipeline_summary

## Step 8: Model data parallel replicas

Data parallelism is the simplest distributed inference strategy when the model fits on one node or shard group. It rarely reduces single-request latency, but it scales throughput and isolates failure domains.

In [ ]:
dp_rows = []
base_request_latency_ms = 980
base_tokens_per_second = 165
for replicas in [1, 2, 4, 8]:
    online_throughput = base_tokens_per_second * replicas * (0.97 if replicas > 1 else 1.0)
    batch_throughput = base_tokens_per_second * replicas * 1.04
    dp_rows.append({
        "replicas": replicas,
        "online_p95_latency_ms": base_request_latency_ms + (18 if replicas > 1 else 0),
        "online_output_tps": round(online_throughput, 2),
        "offline_batch_tps": round(batch_throughput, 2),
        "blast_radius_share": round(1 / replicas, 3),
    })

dp_df = pd.DataFrame(dp_rows)
dp_df

## Step 9: Simulate expert-parallel routing hotspots

MoE and expert-parallel systems can deliver high parameter counts without activating all weights each token. The practical risk is hotspotting: some experts get far more traffic than others and become the real bottleneck.

In [ ]:
def simulate_expert_routing(total_tokens=5000, experts=16, skew=0.28, capacity_factor=1.25):
    raw_weights = np.array([1 + skew * math.cos(index / 2.0) + random.random() * 0.08 for index in range(experts)])
    probabilities = raw_weights / raw_weights.sum()
    assignments = np.random.choice(np.arange(experts), size=total_tokens, p=probabilities)
    counts = Counter(assignments)
    ideal_capacity = total_tokens / experts
    max_capacity = ideal_capacity * capacity_factor
    rows = []
    dropped_tokens = 0
    for expert_id in range(experts):
        load = counts.get(expert_id, 0)
        overload = max(load - max_capacity, 0)
        dropped_tokens += overload
        rows.append({
            "expert_id": expert_id,
            "load_tokens": load,
            "load_ratio_vs_ideal": round(load / ideal_capacity, 3),
            "over_capacity_tokens": int(overload),
        })
    return pd.DataFrame(rows), dropped_tokens

expert_df, dropped_tokens = simulate_expert_routing()
expert_df.head(10)

## Step 10: Visualize expert imbalance

This is the operational question for expert parallelism: are the experts actually sharing work, or is one hot path undoing the promise of sparse activation?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(expert_df["expert_id"].astype(str), expert_df["load_tokens"], color="#4c78a8")
ax.set_title("Expert load distribution")
ax.set_xlabel("expert id")
ax.set_ylabel("routed tokens")
plt.tight_layout()
print("Dropped or rerouted tokens under capacity factor:", int(dropped_tokens))

## Step 11: Compare monolithic serving with prefill-decode disaggregation

A newer practical pattern is splitting long prompt ingestion from token-by-token decode. The idea is simple: keep the prompt-heavy work away from the nodes that need to stay responsive for interactive decode.

In [ ]:
def simulate_disaggregation(long_prompt_tokens, short_prompt_tokens, output_tokens, split_prefill=False):
    monolithic_prefill_ms = 1000 * long_prompt_tokens / 8800
    short_prefill_ms = 1000 * short_prompt_tokens / 8800
    decode_ms = 1000 * output_tokens / 68
    if not split_prefill:
        short_ttft_ms = monolithic_prefill_ms + short_prefill_ms + 18
        total_ms = monolithic_prefill_ms + decode_ms
        return {"mode": "monolithic", "short_ttft_ms": round(short_ttft_ms, 2), "long_request_total_ms": round(total_ms, 2)}
    transfer_ms = 26
    prefill_pool_ms = 1000 * long_prompt_tokens / 12600
    short_ttft_ms = short_prefill_ms + 18
    total_ms = prefill_pool_ms + transfer_ms + decode_ms
    return {"mode": "split_prefill_decode", "short_ttft_ms": round(short_ttft_ms, 2), "long_request_total_ms": round(total_ms, 2)}

disagg_df = pd.DataFrame([
    simulate_disaggregation(long_prompt_tokens=12000, short_prompt_tokens=420, output_tokens=180, split_prefill=False),
    simulate_disaggregation(long_prompt_tokens=12000, short_prompt_tokens=420, output_tokens=180, split_prefill=True),
])
disagg_df

## Step 12: Build a practical decision matrix

Distributed inference choices are workload choices. The table below turns the notebook into advice: when a strategy helps, what it protects, and what new bottleneck it introduces.

In [ ]:
decision_rows = [
    {"workload_shape": "single interactive model that fits", "recommended_strategy": "data_parallel", "protects": "throughput and availability", "watch_out_for": "latency barely improves"},
    {"workload_shape": "model barely exceeds single-node memory", "recommended_strategy": "tensor_parallel", "protects": "memory fit with good online latency", "watch_out_for": "decode-step communication"},
    {"workload_shape": "very large dense model with batch traffic", "recommended_strategy": "pipeline_parallel", "protects": "memory fit and aggregate throughput", "watch_out_for": "pipeline bubbles at low concurrency"},
    {"workload_shape": "sparse MoE with strong router quality", "recommended_strategy": "expert_parallel", "protects": "parameter scale without dense activation", "watch_out_for": "expert hotspots and load imbalance"},
    {"workload_shape": "long-prompt + interactive mixed service", "recommended_strategy": "prefill_decode_split", "protects": "short-request TTFT", "watch_out_for": "KV transfer and orchestration complexity"},
]

decision_df = pd.DataFrame(decision_rows)
decision_df

## Step 13: Export topology-planning artifacts

The exported artifact captures the major tables from this notebook so later operational notebooks can reason about rollout choices or hardware budgets.

In [ ]:
artifact = {
    "notebook": "17_distributed_inference_concepts",
    "fit_table": fit_df.to_dict("records"),
    "tensor_parallel": tp_df.to_dict("records"),
    "pipeline_parallel": pipeline_df.to_dict("records"),
    "data_parallel": dp_df.to_dict("records"),
    "expert_parallel": expert_df.to_dict("records"),
    "prefill_decode_disaggregation": disagg_df.to_dict("records"),
    "decision_matrix": decision_df.to_dict("records"),
}

artifact_path = ARTIFACT_DIR / "distributed_inference_planning.json"
write_json(artifact_path, artifact)
print("Wrote:", artifact_path.resolve())

## Recap

Distributed inference is not one decision. It is a menu of compromises. Tensor parallelism buys memory fit at the price of communication. Pipeline parallelism buys scale at the price of bubbles. Data parallelism buys throughput and fault isolation. Expert parallelism buys sparse scale but risks hotspots. Prefill-decode splitting buys responsiveness for mixed traffic but adds orchestration overhead.

In [ ]:
assert len(tp_df) > 0
assert pipeline_df["bubble_fraction"].max() <= 1.0
assert decision_df["recommended_strategy"].nunique() >= 4
print("Notebook 17 sanity checks passed.")